# SE BGC - Poster Results

Three main result figures built from the full PFT + pigment pipeline. Plots with multiple variants (concentration vs fraction, percentage points vs relative %, pure scatter vs LOWESS overlay) are all presented so the final version can be picked later.

In [ ]:
import sys
from typing import cast
import datetime as dt
from pathlib import Path

ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from eddy_tracking.packages.py_eddy_tracker.observations.tracking import TrackEddiesObservations
from utils.config import resolve_output_dir

EXPERIMENT = "bgc_20241001_20250701"

PFT_COLS = ["Diatoms", "Dinoflagellates", "Haptophytes",
    "Cryptophytes", "Green_algae", "Cyanobacteria"]
FRAC_COLS = [f"{c}_frac" for c in PFT_COLS]

RADIAL_BINS = [0, 0.25, 0.5, 0.75, 1.0, 1.5]
bin_mids = np.array([(RADIAL_BINS[i] + RADIAL_BINS[i + 1]) / 2
    for i in range(len(RADIAL_BINS) - 1)])
bin_labels = [f"{RADIAL_BINS[i]:.2f}-{RADIAL_BINS[i + 1]:.2f}"
    for i in range(len(RADIAL_BINS) - 1)]


def compute_haversine_km(lon1, lat1, lon2, lat2):
    R = 6371.0
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return R * 2 * np.arcsin(np.sqrt(a))


def load_track_props(polarity):
    PET_EPOCH = dt.date(1950, 1, 1)
    zarr_path = resolve_output_dir(EXPERIMENT, "eddy_track", polarity) / f"{polarity}_tracks.zarr"
    tracked = TrackEddiesObservations.load_file(str(zarr_path))
    dates = [PET_EPOCH + dt.timedelta(days=int(d)) for d in tracked.time]
    return pd.DataFrame({
        "track_id": tracked.track.astype(int),
        "date": pd.to_datetime(dates),
        "polarity": polarity,
        "center_lon": (tracked.longitude + 180) % 360 - 180,
        "center_lat": tracked.latitude,
        "radius_km": tracked.radius_e / 1000,
    })


def bootstrap_radial_difference(df, col, metric, n_boot=1000, seed=42):
    """
    Per-bin cyclone-minus-anticyclone difference with 95% bootstrap CIs.

    metric "pct_points": (cyc.mean - anti.mean) * 100
    metric "relative_pct": 200 * (cyc - anti) / (cyc + anti), symmetric and bounded +/-200
    """
    grouped = (df.groupby(["track_id", "date", "r_bin", "polarity"], observed=True)[col]
        .mean().reset_index())
    rng = np.random.default_rng(seed)
    diffs, lo, hi = [], [], []
    for label in bin_labels:
        cyc = grouped.loc[(grouped["r_bin"] == label) & (grouped["polarity"] == "cyclone"), col].values
        anti = grouped.loc[(grouped["r_bin"] == label) & (grouped["polarity"] == "anticyclone"), col].values
        if len(cyc) < 3 or len(anti) < 3:
            diffs.append(np.nan); lo.append(np.nan); hi.append(np.nan)
            continue
        boots = np.empty(n_boot)
        for b in range(n_boot):
            c = rng.choice(cyc, size=len(cyc), replace=True).mean()
            a = rng.choice(anti, size=len(anti), replace=True).mean()
            boots[b] = (c - a) * 100 if metric == "pct_points" else 200 * (c - a) / (c + a)
        cm, am = cyc.mean(), anti.mean()
        diffs.append((cm - am) * 100 if metric == "pct_points" else 200 * (cm - am) / (cm + am))
        lo.append(np.percentile(boots, 2.5))
        hi.append(np.percentile(boots, 97.5))
    return np.array(diffs), np.array(lo), np.array(hi)


pft_frames = []
for polarity in ("cyclone", "anticyclone"):
    pft_dir = resolve_output_dir(EXPERIMENT, "pft", polarity)
    for fp in sorted(pft_dir.glob("eddy_*_pfts.parquet")):
        df = pd.read_parquet(fp)
        df["polarity"] = polarity
        pft_frames.append(df)
pfts = pd.concat(pft_frames, ignore_index=True)

pig_frames = []
for polarity in ("cyclone", "anticyclone"):
    pig_dir = resolve_output_dir(EXPERIMENT, "pigments", polarity)
    for fp in sorted(pig_dir.glob("eddy_*_pigments.parquet")):
        df = pd.read_parquet(fp)
        df["polarity"] = polarity
        pig_frames.append(df)
pigments = pd.concat(pig_frames, ignore_index=True)

track_props = pd.concat(
    [load_track_props(p) for p in ("cyclone", "anticyclone")],
    ignore_index=True,
)
pfts = pfts.merge(
    track_props[["track_id", "date", "polarity", "radius_km"]],
    on=["track_id", "date", "polarity"],
    how="left",
)

pfts["dist_km"] = compute_haversine_km(
    pfts["pixel_lon"].values, pfts["pixel_lat"].values,
    pfts["center_lon"].values, pfts["center_lat"].values,
)
pfts["r_norm"] = pfts["dist_km"] / pfts["radius_km"]

total_pft = pfts[PFT_COLS].sum(axis=1)
for col in PFT_COLS:
    pfts[f"{col}_frac"] = pfts[col] / total_pft

pfts["r_bin"] = pd.cut(pfts["r_norm"], bins=RADIAL_BINS, labels=bin_labels, right=False)
pft_binned = pfts.dropna(subset=["r_bin"])

eddy_means = (
    pfts.groupby(["track_id", "date", "polarity"])[PFT_COLS + FRAC_COLS]
    .mean().reset_index()
)

pig_medians = (
    pigments.groupby(["track_id", "date", "polarity"])["T chla"]
    .mean().reset_index()
)
pig_medians = pig_medians[pig_medians["T chla"] >= 0.01].copy()

print(f"pft_eddy_dates: {len(eddy_means):,}")
print(f"pigment_eddy_dates_t_chla_gte_0_01: {len(pig_medians):,}")

## Plot 1: PFT Distributions by Polarity

Proof-of-concept overlay histograms comparing cyclone (blue) and anticyclone (red) eddies. Each observation is an eddy-date spatial mean, not a raw pixel, to avoid pseudoreplication.

In [ ]:
from matplotlib.patches import Patch

fig, axes = cast("tuple[Figure, np.ndarray]", plt.subplots(2, 3, figsize=(13, 8)))
for i, col in enumerate(PFT_COLS):
    ax = axes.flat[i]
    for pol, color in [("cyclone", "#2166ac"), ("anticyclone", "#b2182b")]:
        vals = eddy_means.loc[eddy_means["polarity"] == pol, col]
        ax.hist(vals, bins=30, alpha=0.5, color=color, density=True)
    ax.set_title(col.replace("_", " "))
    ax.set_xlabel("mg / m³")
    ax.set_ylabel("Probability Density")
    ax.spines[["top", "right"]].set_visible(False)

legend_handles = [
    Patch(facecolor="#2166ac", alpha=0.5, label="Cyclone"),
    Patch(facecolor="#b2182b", alpha=0.5, label="Anticyclone"),
]
fig.tight_layout()
fig.subplots_adjust(top=0.84)
fig.suptitle("PFT Concentration Distributions: Cyclone vs Anticyclone",
    fontsize=13, y=0.95)
fig.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(0.5, 0.925),
    ncol=2, frameon=False, fontsize=11)
fig.savefig("1_pft_concentration_hist.png", dpi=300, bbox_inches="tight")
plt.show()

## Plot 2: Radial PFT Structure

Cyclone − anticyclone difference in PFT fraction across normalized radial bins, expressed as a symmetric relative percent `200 × (cyc − anti) / (cyc + anti)`, with 1000-resample bootstrap 95% CIs.

In [ ]:
fig, axes = cast("tuple[Figure, np.ndarray]", plt.subplots(2, 3, figsize=(13, 8)))
for i, col in enumerate(FRAC_COLS):
    ax = axes.flat[i]
    display_name = col.replace("_frac", "").replace("_", " ")
    diffs, lo, hi = bootstrap_radial_difference(pft_binned, col, metric="relative_pct")
    all_above = np.all(lo[~np.isnan(lo)] > 0)
    all_below = np.all(hi[~np.isnan(hi)] < 0)
    color = "#2166ac" if all_above else "#b2182b" if all_below else "0.4"
    ax.plot(bin_mids, diffs, "o-", color=color, ms=5, lw=1.5)
    ax.fill_between(bin_mids, lo, hi, color=color, alpha=0.2)
    ax.axhline(0, color="0.5", ls="--", lw=0.8)
    ax.set_title(display_name)
    ax.set_xlabel("Normalized Radius (r / R)")
    ax.set_ylabel("Symmetric Relative Difference (%)")
    ax.set_xlim(0, 1.5)
    ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
fig.subplots_adjust(top=0.88)
fig.suptitle("Radial PFT Structure: Cyclones vs. Anticyclone",
    fontsize=13, y=0.96)
fig.savefig("2_radial_pft_structure.png", dpi=300, bbox_inches="tight")
plt.show()

## Plot 3: Mean Chl-a Over Time by Polarity

Per-eddy-halfmonth means aggregated across eddies (one mean per polarity per half-month bin) with 95% bootstrap CIs (2000 resamples). Colored dots: individual eddy-date spatial means, positioned by actual observation date with low alpha so dense regions cumulatively darken. The single anticyclone June outlier (~2.2 mg/m³) is filtered.

In [ ]:
from calendar import monthrange

pig_m = pig_medians.copy()
pig_m["month"] = pig_m["date"].dt.month  # pyright: ignore[reportAttributeAccessIssue]
pig_m["half"] = (pig_m["date"].dt.day > 15).astype(int)  # pyright: ignore[reportAttributeAccessIssue]

# Per-eddy-halfmonth means for the line + CI computation
eddy_hm = (
    pig_m.groupby(["track_id", "polarity", "month", "half"])["T chla"]
    .mean().reset_index()
)

# Remove the anticyclone June outlier (single ~2.2 mg/m^3 point)
eddy_hm = eddy_hm[
    ~((eddy_hm["polarity"] == "anticyclone") & (eddy_hm["T chla"] > 1.5))
].reset_index(drop=True)

MONTH_ORDER = [10, 11, 12, 1, 2, 3, 4, 5, 6]
MONTH_LABELS = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May", "Jun"]

bins_sequence = [(m, h) for m in MONTH_ORDER for h in (0, 1)]
month_tick_positions = [i * 2 + 0.5 for i in range(len(MONTH_ORDER))]


def map_date_to_x(ts):
    """
    Map a date to a continuous x position on the half-month axis. Each bin occupies [center - 0.5, center + 0.5]; the fractional offset within the bin reflects the actual day of month.
    """
    if ts.month not in MONTH_ORDER:
        return np.nan
    month_idx = MONTH_ORDER.index(ts.month)
    if ts.day <= 15:
        bin_center = month_idx * 2
        bin_start, bin_end = 1, 15
    else:
        bin_center = month_idx * 2 + 1
        bin_start = 16
        bin_end = monthrange(ts.year, ts.month)[1]
    within = (ts.day - bin_start) / (bin_end - bin_start) if bin_end > bin_start else 0.5
    return bin_center - 0.5 + within


# Per-eddy-date scatter positioned by actual observation date
scatter_data = pig_medians.copy()
scatter_data["x"] = scatter_data["date"].apply(map_date_to_x)  # pyright: ignore[reportAttributeAccessIssue]
scatter_data = scatter_data.dropna(subset=["x"])  # pyright: ignore[reportCallIssue]
scatter_data = scatter_data[
    ~((scatter_data["polarity"] == "anticyclone") & (scatter_data["T chla"] > 1.5))
]

fig, axes = cast("tuple[Figure, np.ndarray]", plt.subplots(1, 2, figsize=(12, 5.5), sharey=True,
    gridspec_kw={"wspace": 0.08}))

summaries = {}
for pol in ["cyclone", "anticyclone"]:
    means, lo_all, hi_all, ns = [], [], [], []
    for m, h in bins_sequence:
        vals = eddy_hm.loc[
            (eddy_hm["polarity"] == pol)
            & (eddy_hm["month"] == m)
            & (eddy_hm["half"] == h),
            "T chla",
        ].to_numpy()
        if len(vals) == 0:
            means.append(np.nan); lo_all.append(np.nan); hi_all.append(np.nan); ns.append(0)
            continue
        if len(vals) >= 3:
            rng = np.random.default_rng(42)
            boots = np.array([
                np.mean(rng.choice(vals, size=len(vals), replace=True))
                for _ in range(2000)
            ])
            lo, hi = np.percentile(boots, [2.5, 97.5])
        else:
            lo, hi = np.min(vals), np.max(vals)
        means.append(np.mean(vals))
        lo_all.append(lo); hi_all.append(hi); ns.append(len(vals))
    summaries[pol] = (np.array(means), np.array(lo_all), np.array(hi_all), ns)

panel_info = [
    ("cyclone", "#2166ac"),
    ("anticyclone", "#b2182b"),
]

for ax, (pol, color) in zip(axes, panel_info):
    x_positions = np.arange(len(bins_sequence))

    # Scatter at actual date positions (one dot per per-eddy-date observation); low alpha so overlapping observations cumulatively darken dense regions
    sub = scatter_data[scatter_data["polarity"] == pol]
    ax.scatter(sub["x"], sub["T chla"], s=22, color=color, alpha=0.18,
        linewidths=0, zorder=1)

    means, lo_all, hi_all, ns = summaries[pol]
    valid = ~np.isnan(means)
    # lower row then upper row, the two-row layout errorbar expects: two (n_valid_bins,) -> (2, n_valid_bins)
    yerr = np.vstack([means[valid] - lo_all[valid], hi_all[valid] - means[valid]])
    ax.errorbar(x_positions[valid], means[valid], yerr=yerr, fmt="o-",
        color=color, lw=2.8, ms=8,
        capsize=5, capthick=1.8, elinewidth=1.8,
        markeredgecolor="white", markeredgewidth=0.8,
        zorder=3)

    ax.set_xticks(month_tick_positions)
    ax.set_xticklabels(MONTH_LABELS, fontsize=11)
    ax.set_xlabel("Month", fontsize=12, labelpad=8)
    ax.set_title(pol.capitalize(),
        fontsize=13, fontweight="bold", color=color, pad=12)
    ax.tick_params(axis="both", which="major", labelsize=10, length=4, width=1.0)
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_linewidth(1.0)
    ax.grid(axis="y", alpha=0.22, linestyle="--", linewidth=0.6)
    ax.set_axisbelow(True)

y_max = max(eddy_hm["T chla"].max(), scatter_data["T chla"].max()) * 1.06
axes[0].set_ylim(0, y_max)
axes[0].set_ylabel("Mean T chl-a (mg m$^{-3}$)", fontsize=12, labelpad=10)

fig.suptitle("Mean Chl-a Over Time by Polarity",
    fontsize=15, y=1.00, fontweight="bold")
fig.tight_layout()
fig.savefig("3_monthly_tchla_by_polarity.png", dpi=300, bbox_inches="tight")
plt.show()